# 02 — TTT Evaluation on ARC-AGI-2 (Colab Pro)

Loads a slim checkpoint produced by `01_pretrain_colab.ipynb` and runs the eval split with TTT on and off.

Outputs `submission_*.json` in the ARC Prize Kaggle format so the same predictions can be dry-run against a Kaggle notebook later.

In [ ]:
import pathlib
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = pathlib.Path('/content/arc_hybrid_repo')
if not REPO_DIR.exists():
    !git clone https://github.com/Nitish05/ARC-AGI.git {REPO_DIR}

%cd /content/arc_hybrid_repo
!pip install -q -r requirements.txt

In [ ]:
# Pull latest from GitHub (run this whenever you've pushed local edits).
!git -C /content/arc_hybrid_repo pull --ff-only

# Hot-reload so edits to arc_hybrid/*.py take effect on the next cell run
# without a kernel restart. Python 3.12 removed the `imp` module, but Colab's
# bundled autoreload still imports it — shim with importlib.reload first.
import sys, types, importlib
if 'imp' not in sys.modules:
    sys.modules['imp'] = types.ModuleType('imp')
    sys.modules['imp'].reload = importlib.reload
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from arc_hybrid.utils.config import to_namespace
from arc_hybrid.model.hybrid import build_hybrid_from_config
from arc_hybrid.data.arc_loader import filter_max_grid, load_split
from arc_hybrid.eval.evaluate import evaluate_split

# Latest slim checkpoint from pretrain. Kernel disconnected at step ~52150,
# so slim_50000.pt is the last full checkpoint on Drive (ckpt_every=5000).
ckpt_path = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1/slim_50000.pt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
cfg = to_namespace(ckpt['config'])
model = build_hybrid_from_config(cfg).to('cuda')

# Strip torch.compile's `_orig_mod.` prefix from keys if it's there.
# pretrain.py compiles the model before save, so older checkpoints carry it.
state = ckpt['model']
if any(k.startswith('_orig_mod.') for k in state.keys()):
    state = {k.removeprefix('_orig_mod.'): v for k, v in state.items()}
model.load_state_dict(state)
model.eval()
print(f"loaded {sum(p.numel() for p in model.parameters())/1e6:.1f}M params from {ckpt_path}")

In [ ]:
from pathlib import Path
tasks = load_split(Path('data/raw/ARC-AGI-2/data/evaluation'))
tasks = filter_max_grid(tasks, cfg.model.max_grid_size)
print(f'{len(tasks)} eval tasks')

In [ ]:
out_dir = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1/eval'
evaluate_split(model, tasks, use_ttt=False, max_grid=cfg.model.max_grid_size,
               device='cuda', out_dir=out_dir, tag='ttt_off')

In [ ]:
ttt_kwargs = dict(steps=100, lr=5e-4, batch_size=4, n_examples=64,
                  lora_r=8, lora_alpha=16, lora_dropout=0.05)
evaluate_split(model, tasks, use_ttt=True, ttt_kwargs=ttt_kwargs,
               max_grid=cfg.model.max_grid_size, device='cuda',
               out_dir=out_dir, tag='ttt_on')